# Generated Track Visualization

## Generation

In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import scanpy as sc
import seaborn as sns
from sklearn.neighbors import NearestNeighbors
from scipy.interpolate import interp1d
import h5py
from scipy.sparse import csr_matrix
import os
import json

OUTDIR = "deetyabn/outputs_diffusion_10k_vae5_"
os.makedirs(OUTDIR, exist_ok=True)

from torch.utils.data import Dataset, DataLoader
from multimodal_dataset import MultimodalDataManager
from tqdm import tqdm
from sklearn.metrics import mean_squared_error, r2_score

In [2]:
dm = MultimodalDataManager("pseudobulking", suffix="10k")

train_ids = np.load("./pseudobulking/train_ids_10k.npy", allow_pickle=True)
val_ids   = np.load("./pseudobulking/val_ids_10k.npy", allow_pickle=True)
test_ids  = np.load("./pseudobulking/test_ids_10k.npy", allow_pickle=True)
held_ids  = np.load("./pseudobulking/held_out_ids_10k.npy", allow_pickle=True)

train_ids = train_ids.astype(str)
val_ids   = val_ids.astype(str)
test_ids  = test_ids.astype(str)
held_ids  = held_ids.astype(str)

In [3]:
def cosine_noise_schedule(T, s=0.008):
  # define alpha_t as cosine curve
  x = torch.linspace(0, T, T+1)
  alpha_bar_prod = torch.cos(((x/T) + s) / (1+s) * torch.pi/2) ** 2 # fraction of signal left after t steps (cosine with slight offset)
  alpha_bar_prod = alpha_bar_prod/alpha_bar_prod[0] # correct for offset so a_0 = 1
  betas = 1 - (alpha_bar_prod[1:] / alpha_bar_prod[:-1])
  return torch.clamp(betas, min=1e-5, max=0.999)

# diffusion portion
class ATACDiffusion:
  def __init__(self, T=1000, device='cuda'):
    self.T = T
    self.device = device
    # precompute noise scale, signal scale at each time step
    betas = cosine_noise_schedule(T).to(device)
    alphas = 1.0 - betas
    self.betas = betas
    self.alphas = alphas
    self.alpha_bar = torch.cumprod(alphas, dim=0)
    self.sqrt_alpha_cp = self.alpha_bar.sqrt()
    self.sqrt_noise_scale = (1.0 - self.alpha_bar).sqrt()

    # used later for p sampling
    prev_alpha_bar = torch.cat([torch.ones(1).to(device), self.alpha_bar[:-1]])
    self.posterior_variance = betas * (1.0 - prev_alpha_bar) / (1.0 - self.alpha_bar)

  # forward: add noise
  def q_sample(self, x_0, t, noise=None):
    if noise is None:
      noise = torch.randn_like(x_0)
    sqrt_alpha = self.sqrt_alpha_cp[t].view(-1, 1)
    sqrt_noise_sc = self.sqrt_noise_scale[t].view(-1, 1)
    return sqrt_alpha * x_0 + sqrt_noise_sc * noise, noise

  # backward: remove noise using denoising model
  @torch.no_grad()
  def p_sample(self, model, xt, z, t):
    t_tensor = torch.full((xt.shape[0],), t, device=self.device, dtype=torch.long)
    eps_pred = model(xt, z, t_tensor)
    alpha = self.alphas[t]
    mean = (1 / alpha.sqrt()) * (xt - (self.betas[t] / self.sqrt_noise_scale[t]) * eps_pred)
    if t == 0:
        return mean
    return mean + self.posterior_variance[t].sqrt() * torch.randn_like(xt)

  # repeat denoising for T time steps
  @torch.no_grad()
  def sample(self, model, z, atac_dim, n_samples=1):
    x = torch.randn(n_samples, atac_dim).to(self.device)
    for t in reversed(range(self.T)):
      x = self.p_sample(model, x, z, t)
    return x

  @torch.no_grad()
  def ddim_sample(self, model, z, atac_dim, n_samples=1, steps=50):
      x = torch.randn(n_samples, atac_dim).to(self.device)
      step_indices = torch.linspace(self.T - 1, 0, steps).long()
      for i, t in enumerate(step_indices):
          t_tensor = torch.full((n_samples,), t, device=self.device, dtype=torch.long)
          eps = model(x, z, t_tensor)
          alpha_bar = self.alpha_bar[t]
          x0_pred = (x - (1 - alpha_bar).sqrt() * eps) / alpha_bar.sqrt()
          x0_pred = torch.clamp(x0_pred, -5, 5)
          if i < len(step_indices) - 1:
              t_next = step_indices[i + 1]
              alpha_bar_next = self.alpha_bar[t_next]
              x = alpha_bar_next.sqrt() * x0_pred + (1 - alpha_bar_next).sqrt() * eps
          else:
              x = x0_pred
      return x

# map timesteps to continuous vector to be meaningful for model
class SinusoidalEmbedding(nn.Module):
  def __init__(self, dim):
    super().__init__()
    self.dim = dim

  def forward(self, t):
    device = t.device
    half = self.dim // 2
    freqs = torch.exp(-np.log(10000) * torch.arange(half, device=device) / half)
    args = t[:, None].float() * freqs[None]
    return torch.cat([args.sin(), args.cos()], dim=-1)

# core residual block for denoiser
class ResidualBlock(nn.Module):
  def __init__(self, dim, cond_dim):
    super().__init__()
    self.net = nn.Sequential(
            nn.LayerNorm(dim),
            nn.Linear(dim, dim * 2),
            nn.SiLU(),
            nn.Linear(dim * 2, dim),
      )
    # project conditional vector (z = RNAseq latent embedding from Schema + t from timestep)
    self.cond_proj = nn.Linear(cond_dim, dim)

  def forward(self, x, cond):
    return x + self.net(x + self.cond_proj(cond))

# denoiser
class ATACDenoiser(nn.Module):
  def __init__(self, atac_dim, z_dim=64, hidden=512, depth=6, t_emb_dim=128):
    super().__init__()

    self.t_emb = SinusoidalEmbedding(t_emb_dim)
    cond_dim = z_dim + t_emb_dim
    self.cond_proj = nn.Sequential( # combine z + t to single conditioning vec, project to hidden dim
            nn.Linear(cond_dim, hidden),
            nn.SiLU(),
            nn.Linear(hidden, hidden),
    )

    # project input to hidden dimension, compress atac peaks
    self.input_proj = nn.Linear(atac_dim, hidden)

    # denoising core with residual blocks
    self.blocks = nn.ModuleList([
            ResidualBlock(hidden, hidden) for _ in range(depth)
    ])
    self.output = nn.Sequential(
            nn.LayerNorm(hidden),
            nn.Linear(hidden, atac_dim),
    )

  def forward(self, xt, z, t):
    t_emb = self.t_emb(t)
    cond = self.cond_proj(torch.cat([z, t_emb], dim=-1))
    h = self.input_proj(xt)
    for block in self.blocks:
      h = block(h, cond)
    return self.output(h)

In [4]:
embeddings_df = pd.read_csv('/orcd/data/manoli/001/tnikitha/scAD/vae_latent_space.csv')
embeddings_df["Sample_barcode"] = embeddings_df["Sample_barcode"].astype(str)
latent_cols = [c for c in embeddings_df.columns if c.startswith("latent_")]
X = embeddings_df[latent_cols].values
X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)
mean = X.mean(axis=0)
std = X.std(axis=0)
std[std < 1e-6] = 1  # prevent division issues
X = (X - mean) / std
embeddings_df[latent_cols] = X
embeddings_indexed = embeddings_df.set_index("Sample_barcode")

class VAEEmbeddingDataset(Dataset):
    def __init__(self, base_dataset, embeddings_indexed, ids, latent_cols):
        self.base_dataset = base_dataset
        self.embeddings_indexed = embeddings_indexed
        self.ids = np.asarray(ids).astype(str)
        self.latent_cols = latent_cols

        assert len(self.base_dataset) == len(self.ids), (
            f"Dataset length {len(self.base_dataset)} != IDs length {len(self.ids)}"
        )

        missing = set(self.ids) - set(self.embeddings_indexed.index)
        if len(missing) > 0:
            raise ValueError(f"{len(missing)} IDs missing from embeddings_df.")

    def __len__(self):
        return len(self.base_dataset)

    def __getitem__(self, idx):
        item = self.base_dataset[idx]

        barcode = self.ids[idx]

        x = self.embeddings_indexed.loc[barcode, self.latent_cols].values.astype("float32")
        y = item["atac"].float()

        return {
            "rna_emb": torch.from_numpy(x),
            "atac": y,
            "barcode": barcode,
        }


In [5]:
train_ds = VAEEmbeddingDataset(
    dm.get_dataset("train"),
    embeddings_indexed,
    train_ids,
    latent_cols,
)

val_ds = VAEEmbeddingDataset(
    dm.get_dataset("val"),
    embeddings_indexed,
    val_ids,
    latent_cols,
)

test_ds = VAEEmbeddingDataset(
    dm.get_dataset("test"),
    embeddings_indexed,
    test_ids,
    latent_cols,
)

held_ds = VAEEmbeddingDataset(
    dm.get_dataset("held_out"),
    embeddings_indexed,
    held_ids,
    latent_cols,
)

In [6]:
train_loader = DataLoader(train_ds, batch_size=64, shuffle=True)
val_loader   = DataLoader(val_ds, batch_size=128, shuffle=False)
test_loader  = DataLoader(test_ds, batch_size=128, shuffle=False)
held_loader  = DataLoader(held_ds, batch_size=128, shuffle=False)

In [7]:
sample_batch = next(iter(train_loader))

print(sample_batch["rna_emb"].shape)
print(sample_batch["atac"].shape)
print(sample_batch["barcode"][:5])

input_size = sample_batch["rna_emb"].shape[1]
output_size = sample_batch["atac"].shape[1]

print("Input size:", input_size)
print("Output size:", output_size)


torch.Size([64, 30])
torch.Size([64, 1010153])
[np.str_('230211Kel_3#CACATTAAGGTCCAAT-1'), np.str_('231019KelA_AT1#GCAGCAACACAATACT-1'), np.str_('231019KelA_EC1#TTTCACCCAACACCTA-1'), np.str_('231019KelA_MT8#GGGCATTGTTTATTCG-1'), np.str_('230211Kel_14#TAACAAGCACGTTACA-1')]
Input size: 30
Output size: 1010153


In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
atac_dim = sample_batch["atac"].shape[1]
diffusion = ATACDiffusion(device=device)
model = ATACDenoiser(atac_dim, z_dim=30)
model = model.to(device)

In [9]:
model.load_state_dict(torch.load(f"{OUTDIR}/best_10k_model.pt"))

<All keys matched successfully>

In [10]:
preds = []
targets = []
barcodes = []

model.eval()
n_sample_cells = 64

with torch.no_grad():
    collected = 0
    for batch in train_loader:
        if collected >= n_sample_cells:
            break

        x = batch["rna_emb"].to(device)
        y = torch.log1p(batch["atac"].to(device))
        bc = batch["barcode"]

        pred = diffusion.ddim_sample(model, x, y.shape[1], n_samples=x.shape[0], steps=50)

        preds.append(pred.cpu().numpy())
        targets.append(y.cpu().numpy())
        barcodes.append(bc)
        collected += x.shape[0]

In [54]:
print(preds[0][50])
print(targets[0][50])
print(targets[0].shape)

[ 4.7374315  4.764367  -4.957812  ...  4.753729  -4.8289027  4.7074065]
[0.        1.0986123 0.        ... 0.        0.        0.       ]
(64, 1010153)


In [ ]:
mean_baseline = np.mean(targets[0], axis=0)
print(mean_baseline.shape)

(1010153,)


In [57]:
np.save(f"/orcd/data/manoli/001/tnikitha/scAD/predicted_track.npy", preds[0][0])
np.save(f"/orcd/data/manoli/001/tnikitha/scAD/observed_track.npy", targets[0][0])
np.save(f"/orcd/data/manoli/001/tnikitha/scAD/mean_training_track.npy", mean_baseline)

# Visualization

In [60]:
import argparse
import numpy as np
import pandas as pd
import h5py
import scipy.sparse as sp
import pyBigWig

In [61]:
H5_PATH          = "/home/tnikitha/orcd/scratch/pseudobulk_atac_stratified_10k.h5"
PREDICTED_NPY    = "predicted_track.npy"
OBSERVED_NPY     = "observed_track.npy"
BASELINE_NPY     = "mean_training_track.npy"
OUT_PREDICTED    = "predicted.bw"
OUT_OBSERVED     = "observed.bw"
OUT_BASELINE     = "baseline.bw"

In [62]:
# parse peaks from sparse matrix
def parse_peaks(peak_names):
    peaks_parsed = []
    for peak in peak_names:
        parts = peak.split(':')
        chrom = parts[0]
        pos = parts[1].split('-')
        start = int(pos[0])
        end = int(pos[1])
        peaks_parsed.append({'chrom': chrom, 'start': start, 'end': end, 'peak': peak})
    return pd.DataFrame(peaks_parsed)

In [63]:
atac_multi_sampled = sc.read_h5ad("./pseudobulking/atac_multi_sampled_10k.h5ad")
coords = parse_peaks(atac_multi_sampled.var.index)

In [64]:
chrom_sizes_df = pd.read_csv("/orcd/data/manoli/001/tnikitha/datasets/genomes/hg38/hg38.chrom.sizes.main", sep="\t", header=None, names=["chrom", "size"])
chrom_sizes = chrom_sizes_df.set_index("chrom")["size"].astype(int).to_dict()

In [65]:
def write_bw(signal: np.ndarray, coords: pd.DataFrame,
             chrom_sizes: dict, out_path: str):
    """Write a dense peak signal array to a bigWig file."""
    coords = coords.copy()
    coords['value'] = signal

    # sort by chrom order matching the header, then by start position within each chrom
    chrom_order = {chrom: i for i, chrom in enumerate(chrom_sizes.keys())}
    coords['chrom_rank'] = coords['chrom'].map(chrom_order)
    coords_sorted = coords.sort_values(['chrom_rank', 'start']).drop(columns='chrom_rank').reset_index(drop=True)

    bw = pyBigWig.open(out_path, "w")
    bw.addHeader(list(chrom_sizes.items()))

    bw.addEntries(
        coords_sorted['chrom'].tolist(),
        coords_sorted['start'].tolist(),
        ends=coords_sorted['end'].tolist(),
        values=coords_sorted['value'].tolist(),
    )
    bw.close()
    print(f"  wrote {out_path}")

In [66]:
predicted = np.load(PREDICTED_NPY).astype(np.float32)

observed = np.load(OBSERVED_NPY).astype(np.float32)

baseline = np.load(BASELINE_NPY).astype(np.float32)

In [67]:
write_bw(predicted, coords, chrom_sizes, OUT_PREDICTED)
write_bw(observed,  coords, chrom_sizes, OUT_OBSERVED)
write_bw(baseline, coords, chrom_sizes, OUT_BASELINE)

  wrote predicted.bw
  wrote observed.bw
  wrote baseline.bw


In [68]:
!pyGenomeTracks \
  --tracks diffusion_generated_track.ini \
  --region chr11:116000000-117000000 \
  --trackLabelFraction 0.07 \
  --outFileName diffusion_generated_tracks_vis.png

INFO:pygenometracks.tracksClass:initialize 1. [Predicted]
INFO:pygenometracks.tracksClass:initialize 2. [spacer]
INFO:pygenometracks.tracksClass:initialize 3. [Baseline]
INFO:pygenometracks.tracksClass:initialize 4. [spacer]
INFO:pygenometracks.tracksClass:initialize 5. [Observed]
INFO:pygenometracks.tracksClass:initialize 6. [spacer]
INFO:pygenometracks.tracksClass:initialize 7. [x-axis]
INFO:pygenometracks.tracksClass:time initializing track(s):
INFO:pygenometracks.tracksClass:0.001115560531616211
DEBUG:pygenometracks.tracksClass:Figure size in cm is 40 x 11.569148936170214. Dpi is set to 72

INFO:pygenometracks.tracksClass:plotting 1. [Predicted]
INFO:pygenometracks.tracksClass:plotting 2. [spacer]
INFO:pygenometracks.tracksClass:plotting 3. [Baseline]
INFO:pygenometracks.tracksClass:plotting 4. [spacer]
INFO:pygenometracks.tracksClass:plotting 5. [Observed]
INFO:pygenometracks.tracksClass:plotting 6. [spacer]
INFO:pygenometracks.tracksClass:plotting 7. [x-axis]
